# Image Colorization (Regression) with Skip Connections

This notebook trains a CNN to map grayscale images to RGB using MSE loss. Includes a U-Net style skip connection for better detail preservation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import layers, Model, Input

In [ ]:
(x_train, _), (x_test, _) = cifar10.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Convert to grayscale
def rgb2gray(images):
    gray = np.dot(images[...,:3], [0.2989, 0.5870, 0.1140])
    return gray.reshape(-1, 32, 32, 1)

x_train_gray = rgb2gray(x_train)
x_test_gray = rgb2gray(x_test)

In [ ]:
inputs = Input(shape=(32,32,1))

# Encoder
c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
c2 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1)

# Bottleneck
b = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c2)

# Decoder
u = layers.Conv2D(64, (3,3), activation='relu', padding='same')(b)

# Skip connection
merge = layers.Concatenate()([u, c2])

outputs = layers.Conv2D(3, (3,3), activation='sigmoid', padding='same')(merge)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse')

model.summary()

In [ ]:
history = model.fit(
    x_train_gray, x_train,
    validation_data=(x_test_gray, x_test),
    epochs=10,
    batch_size=64
)

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Loss Curve')
plt.show()

In [ ]:
preds = model.predict(x_test_gray[:5])

for i in range(5):
    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1)
    plt.title('Gray')
    plt.imshow(x_test_gray[i].reshape(32,32), cmap='gray')
    plt.subplot(1,3,2)
    plt.title('Predicted')
    plt.imshow(preds[i])
    plt.subplot(1,3,3)
    plt.title('Original')
    plt.imshow(x_test[i])
    plt.show()